In [1]:
import os
import pandas as pd

from pathlib import Path



In [2]:
#set current working directory to the library path
library_path = Path('/villa/rhh25/Cellhit/') #replace with yours
os.chdir(library_path)
#sys.path.append(library_path) use this in case it is still not working

In [ ]:
from CellHit.LLMs import fetch_abstracts, generate_prompt
from CellHit.data import obtain_drugs_metadata
from CellHit.data import prepare_data


In [3]:
#specify the data_path
data_path = library_path/'data'
describe_prompt_path = data_path/'prompts'/'drug_description_prompt.txt'
refine_prompt_path = data_path/'prompts'/'drug_refiner_prompt.txt'
model_path = Path('~/mixtral/').expanduser() # note that the model needs to be downloaded from huggingface, the one used can be found at https://huggingface.co/TheBloke/Mixtral-8x7B-Instruct-v0.1-GPTQ (gptq-4bit-32g-actorder_True)

In [5]:
dataset = 'gdsc'
metadata_gdsc = obtain_drugs_metadata(dataset,data_path)
metadata_gdsc.head()

,DrugID,Drug,PUTATIVE_TARGET,MOA,DRUG_SYNONYMS,SMILES,repurposing_target
0,1003,Camptothecin,TOP1,DNA replication,"Camptothecine, (+)-Camptothecin",CC[C@@]1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=...,TOP1
1,1004,Vinblastine,Microtubule destabiliser,Mitosis,Velban,CC[C@@]1(C[C@H]2C[C@@](C3=C(CCN(C2)C1)C4=CC=CC...,TUBB
2,1005,Cisplatin,DNA crosslinker,DNA replication,"cis-Diammineplatinum(II) dichloride, Platinol,...",N.N.Cl[Pt]Cl,DNA
3,1006,Cytarabine,Antimetabolite,Other,"Ara-Cytidine, Arabinosyl Cytosine, U-19920",C1=CN(C(=O)N=C1N)[C@H]2[C@H]([C@@H]([C@H](O2)C...,DNA synthesis
4,1007,Docetaxel,Microtubule stabiliser,Mitosis,"RP-56976, Taxotere",CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@...,TUBB


In [151]:
dataset = 'prism'
metadata_prism = obtain_drugs_metadata(dataset,data_path)
metadata_prism

,Drug,BroadID,DrugID,MOA,repurposing_target
0,"1,12-BESM",BRD:BRD-K04603573-376-01-3,0,POLYAMINE BIOSYNTHESIS INHIBITOR,NaN
1,"1,3-DIPROPYL-8-PHENYLXANTHINE",BRD:BRD-K41853443-001-02-9,1,ADENOSINE RECEPTOR ANTAGONIST,NaN
2,"1,4-BUTANEDIOL",BRD:BRD-K22346679-001-06-3,2,"BENZODIAZEPINE RECEPTOR AGONIST, GAMMA HYDROXY...","MAN1B1, PLA2G2A, PLA2G2E"
3,"1,5-DICAFFEOYLQUINIC-ACID",BRD:BRD-K53083666-001-01-0,3,NaN,NaN
4,"1-((Z)-3-CHLOROALLYL)-1,3,5,7-TETRAAZAADAMANTA...",BRD:BRD-K88956297-003-01-9,4,NaN,NaN
...,...,...,...,...,...
6426,ZM-336372,BRD:BRD-K73789395-001-10-7,6314,RAF INHIBITOR,"BRAF, LCK, MAPK14, RAF1"
6427,ZOFENOPRIL-CALCIUM,BRD:BRD-K46654563-238-01-0,6317,ANGIOTENSIN CONVERTING ENZYME INHIBITOR,ACE
6428,ZOLPIDEM,BRD:BRD-K44876623-001-03-7,6323,BENZODIAZEPINE RECEPTOR AGONIST,"GABRA1, GABRA2, GABRA3"
6429,ZONIPORIDE,BRD:BRD-K43992824-300-01-0,6326,SODIUM/HYDROGEN EXCHANGER INHIBITOR,SLC9A1


In [158]:
metadata_prism = metadata_prism.dropna(subset=['repurposing_target', 'MOA'])
metadata_prism


,Drug,BroadID,DrugID,MOA,repurposing_target
2,"1,4-BUTANEDIOL",BRD:BRD-K22346679-001-06-3,2,"BENZODIAZEPINE RECEPTOR AGONIST, GAMMA HYDROXY...","MAN1B1, PLA2G2A, PLA2G2E"
7,1-AZAKENPAULLONE,BRD:BRD-K03273112-001-01-8,7,GLYCOGEN SYNTHASE KINASE INHIBITOR,"CCNB1, CDK1, CDK5, GSK3B"
8,1-EBIO,BRD:BRD-K70586315-001-02-4,9,POTASSIUM CHANNEL ACTIVATOR,"KCNN1, KCNN2, KCNN3, KCNN4"
12,1-PHENYLBIGUANIDE,BRD:BRD-K31491153-003-05-9,14,SEROTONIN RECEPTOR AGONIST,"HTR3A, HTR3B"
14,10-DEBC,BRD:BRD-K70792160-003-04-6,16,AKT INHIBITOR,PIM1
...,...,...,...,...,...
6426,ZM-336372,BRD:BRD-K73789395-001-10-7,6314,RAF INHIBITOR,"BRAF, LCK, MAPK14, RAF1"
6427,ZOFENOPRIL-CALCIUM,BRD:BRD-K46654563-238-01-0,6317,ANGIOTENSIN CONVERTING ENZYME INHIBITOR,ACE
6428,ZOLPIDEM,BRD:BRD-K44876623-001-03-7,6323,BENZODIAZEPINE RECEPTOR AGONIST,"GABRA1, GABRA2, GABRA3"
6429,ZONIPORIDE,BRD:BRD-K43992824-300-01-0,6326,SODIUM/HYDROGEN EXCHANGER INHIBITOR,SLC9A1


In [4]:

df_drugcomb = pd.read_csv(data_path/'metadata'/'drugcombs_scored.csv')

In [5]:
## Check if the drugs in drugcomb are in the Prism metadata
drugcomb_drugs = set(df_drugcomb['Drug1']).union(set(df_drugcomb['Drug2']))
prism_drugs = set(metadata_prism['Drug'])



NameError: name 'metadata_prism' is not defined

In [6]:
len(drugcomb_drugs)

6245

In [162]:
drugcomb_to_prism_cleaned = drugcomb_prism_mapping(drugcomb_drugs, prism_drugs)

In [175]:
metadata_prism[metadata_prism['Drug'].isin(list(drugcomb_to_prism_cleaned.values()))].reset_index(drop=True)

,Drug,BroadID,DrugID,MOA,repurposing_target
0,10-HYDROXYCAMPTOTHECIN,BRD:BRD-K63784565-001-06-2,17,TOPOISOMERASE INHIBITOR,TOP1
1,2-DEOXYGLUCOSE,BRD:BRD-K97808269-001-02-7,40,GLYCOLYSIS INHIBITOR,"SLC2A1, SLC2A2, SLC2A3, SLC2A4"
2,2-HYDROXYFLUTAMIDE,BRD:BRD-K82027074-001-07-5,46,ANDROGEN RECEPTOR ANTAGONIST,AR
3,2-IMINOBIOTIN,BRD:BRD-K07954936-001-01-3,48,NITRIC OXIDE SYNTHASE INHIBITOR,"NOS1, NOS2"
4,2-METHOXYESTRADIOL,BRD:BRD-K44408410-001-17-6,51,HYPOXIA INDUCIBLE FACTOR INHIBITOR,"COMT, CYP19A1, CYP1A1, CYP1B1, HIF1A, TUBB"
...,...,...,...,...,...
1822,Y-39983,BRD:BRD-K56751279-300-01-4,6249,RHO ASSOCIATED KINASE INHIBITOR,"ROCK1, ROCK2"
1823,YK-4-279,BRD:BRD-A62182663-001-03-0,6252,APOPTOSIS INHIBITOR,"EWSR1, FLI1"
1824,YOHIMBINE,BRD:BRD-K35586044-003-15-4,6267,ADRENERGIC RECEPTOR ANTAGONIST,"ADRA2A, ADRA2B, ADRA2C, DRD2, DRD3, HTR1A, HTR..."
1825,ZM-336372,BRD:BRD-K73789395-001-10-7,6314,RAF INHIBITOR,"BRAF, LCK, MAPK14, RAF1"


In [ ]:
drug_list = list(drugcomb_to_prism_cleaned.values())

1827

In [170]:
1827/5900

0.30966101694915255

In [101]:
drugcomb_to_prism_keys = set(drugcomb_to_prism_.keys())
drugcomb_to_prism_cleaned_keys = set(drugcomb_to_prism_cleaned.keys())

diff_keys = drugcomb_to_prism_keys - drugcomb_to_prism_cleaned_keys

In [102]:
len(diff_keys)

233

In [105]:
sub_dict = {k: drugcomb_to_prism_[k] for k in diff_keys if k in drugcomb_to_prism_}


In [106]:
sub_dict

{'hg-10-102-01': 'G-1',
 'NVP-BGT226': 'BGT226',
 'Paricalcitol-D6': 'PARICALCITOL',
 'I-BET151 (GSK1210151A)': 'I-BET151',
 'Y-27632 2HCL': 'Y-27632',
 'Ilomastat (GM6001- Galardin)': 'ILOMASTAT',
 'Phortress (NSC-710305)': 'PHORTRESS',
 "6-Bromoindirubin-3'-acetoxime": 'INDIRUBIN',
 'NSC-127716': 'C-1',
 '17-hydroxy Wortmannin': 'WORTMANNIN',
 'PALBOCICLIB (PD-0332991) HCL': 'PALBOCICLIB',
 '(-)-PARTHENOLIDE': 'PARTHENOLIDE',
 'ML130 (NODINITIB-1)': 'ML130',
 'KU-55933 (ATM KINASE INHIBITOR)': 'KU-55933',
 '17-Methyltestosterone': 'TESTOSTERONE',
 'REGORAFENIB (BAY 73-4506)': 'REGORAFENIB',
 'RomidepsinFK-228': 'ROMIDEPSIN',
 '2-Cl-IB-Meca': 'IB-MECA',
 'Darolutamide (ODM-201)': 'ODM-201',
 'MOTESANIB DIPHOSPHATE (AMG-706)': 'MOTESANIB',
 'BGJ398 (NVP-BGJ398)': 'NVP-BGJ398',
 'BRIVANIB (BMS-540215)': 'BRIVANIB',
 'AG-1478': 'G-1',
 'CEP-18770 (DELANZOMIB)': 'DELANZOMIB',
 'AEE788 (NVP-AEE788)': 'AEE788',
 '1-Methyl-D-tryptophan': 'TRYPTOPHAN',
 'S-(+)-Rolipram': 'ROLIPRAM',
 'BMS-4':

In [17]:
shape_base = df_drugcomb.shape[0]
shape_overlap = df_drugcomb[df_drugcomb['Durg1_Upper'].isin(prism_drugs) & df_drugcomb['Durg2_Upper'].isin(prism_drugs)].shape[0]



In [19]:
shape_overlap/shape_base

0.3230272719072294

In [83]:
import requests

def get_chembl_id(drug_name):
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
    r = requests.get(url).json()
    
    if r["molecules"]:
        return r["molecules"][0]["molecule_chembl_id"], r
    return None

chembl_id, r = get_chembl_id("(+)-BICUCULLINE")

print(chembl_id)

CHEMBL515679


In [9]:
def get_moa_chembl(drug_name: str):
    """Fallback: ChEMBL targets (what DrugComb originally used)."""
    try:
        molecule = new_client.molecule
        target = new_client.target
        # Search molecule by name
        res = molecule.search(drug_name)
        if not res:
            return {"source": "ChEMBL", "status": "not_found", "moa": None}
        
        chembl_id = res[0]['molecule_chembl_id']
        # Get bioactivity (targets with mechanism)
        activities = new_client.activity.filter(molecule_chembl_id=chembl_id, standard_type='IC50')
        targets_found = set()
        for act in activities[:10]:  # top 10 activities
            if act.get('target_chembl_id'):
                t = target.get(act['target_chembl_id'])
                if t and t.get('pref_name'):
                    targets_found.add(t['pref_name'])
        
        return {
            "source": "ChEMBL",
            "status": "success",
            "chembl_id": chembl_id,
            "moa": list(targets_found) if targets_found else "No targets found"
        }
    except Exception as e:
        return {"source": "ChEMBL", "status": "error", "moa": str(e)}


In [10]:
get_moa_chembl("(+)-BICUCULLINE")

{'source': 'ChEMBL',
 'status': 'success',
 'chembl_id': 'CHEMBL515679',
 'moa': 'No targets found'}

In [81]:
# def get_moa_from_chembl_id(chembl_id):
#     #url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism.json?molecule_chembl_id={chembl_id}"
#     url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}.json"
#     r = requests.get(url).json()
    
#     results = []
#     print(r)
#     for m in r.get("mechanisms", []):
#         results.append({
#             "moa": m.get("mechanism_of_action"),
#             "target": m.get("target_chembl_id"),
#             "action": m.get("action_type")
#         })
#     return results

# print(get_moa_from_chembl_id(chembl_id))
def get_moa_for_drug(chembl_id):
    """
    Retrieve mechanism of action for a given ChEMBL ID.
    Returns a string combining all mechanism actions, or None if none found.
    """
    url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism?filter=compound_chembl_id:{chembl_id}&format=json"

    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            mechanisms = data.get('mechanisms', [])
            if mechanisms:
                # Extract the mechanism_of_action or action_type (prefer mechanism_of_action)
                moas = []
                for mech in mechanisms:
                    moa = mech.get('mechanism_of_action')
                    if not moa:
                        # Fallback to action_type
                        moa = mech.get('action_type')
                    if moa:
                        moas.append(moa)
                if moas:
                    # Join multiple MOAs with semicolon
                    return '; '.join(set(moas))
    except Exception:
        pass
    return None

In [3]:
import pandas as pd
import os
import time
from tqdm import tqdm
import pubchempy as pcp
from chembl_webresource_client.new_client import new_client
import json
    # will be downloaded automatically
OUTPUT_CSV = "drugcomb_drugs_with_moa.csv"
CACHE_FILE = "drugcomb_moa_cache.json"       #


# ====================== HELPER FUNCTIONS ======================
def get_moa_pubchem(drug_name: str):
    """Primary source: PubChem pharmacological actions / MeSH classifications."""
    try:
        compounds = pcp.get_compounds(drug_name, 'name')
        if not compounds:
            return {"source": "PubChem", "status": "not_found", "moa": None}
        
        cid = compounds[0].cid
        # Pharmacological actions (best MOA proxy)
        props = pcp.get_properties(['PharmacologicalAction', 'MeshHeading'], cid, namespace='cid')
        if props:
            action = props[0].get('PharmacologicalAction')
            mesh = props[0].get('MeshHeading')
            return {
                "source": "PubChem",
                "status": "success",
                "cid": cid,
                "moa": action or mesh or "No pharmacological action listed"
            }
        return {"source": "PubChem", "status": "no_action", "moa": None}
    except Exception as e:
        return {"source": "PubChem", "status": "error", "moa": str(e)}

def get_moa_chembl(drug_name: str):
    """Fallback: ChEMBL targets (what DrugComb originally used)."""
    try:
        molecule = new_client.molecule
        target = new_client.target
        # Search molecule by name
        res = molecule.search(drug_name)
        if not res:
            return {"source": "ChEMBL", "status": "not_found", "moa": None}
        
        chembl_id = res[0]['molecule_chembl_id']
        # Get bioactivity (targets with mechanism)
        activities = new_client.activity.filter(molecule_chembl_id=chembl_id, standard_type='IC50')
        targets_found = set()
        for act in activities[:10]:  # top 10 activities
            if act.get('target_chembl_id'):
                t = target.get(act['target_chembl_id'])
                if t and t.get('pref_name'):
                    targets_found.add(t['pref_name'])
        
        return {
            "source": "ChEMBL",
            "status": "success",
            "chembl_id": chembl_id,
            "moa": list(targets_found) if targets_found else "No targets found"
        }
    except Exception as e:
        return {"source": "ChEMBL", "status": "error", "moa": str(e)}



In [69]:

import pandas as pd
import requests
import time
import re
from tqdm import tqdm 

In [70]:
# -------------------------------------------------------------------
# Helper functions for PubChem queries
# -------------------------------------------------------------------
def is_cas_number(text):
    """Return True if text looks like a CAS Registry Number (e.g., 483-60-3)."""
    return bool(re.match(r'^\d{1,7}-\d{2}-\d{1}$', text.strip()))

def get_cid_by_name(name):
    """Get PubChem CID for a given name (or CAS number via synonym)."""
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/cids/JSON"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            cids = data.get('IdentifierList', {}).get('CID', [])
            return cids[0] if cids else None
    except Exception:
        pass
    return None

def get_cid_by_cas(cas):
    """Get PubChem CID via the xref endpoint for CAS numbers."""
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/xref/RegistryID/{cas}/cids/JSON"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            cids = data.get('IdentifierList', {}).get('CID', [])
            return cids[0] if cids else None
    except Exception:
        pass
    return None

def get_moa_from_cid(cid):
    """Retrieve Mechanism of Action (first Pharmacological Action) for a given CID."""
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/description/JSON"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            info_list = data.get('InformationList', {}).get('Information', [])
            for info in info_list:
                actions = info.get('PharmacologicalAction', [])
                if actions:
                    # Return the first action's description; you may join multiple
                    return actions[0].get('Action', 'Unknown')
    except Exception:
        pass
    return None

def get_moa_for_drug(drug_str, cache):
    """Main function: get MOA for a drug identifier using caching."""
    if drug_str in cache:
        return cache[drug_str]

    moa = None
    # Try direct name lookup first (works for most names and CAS synonyms)
    cid = get_cid_by_name(drug_str)
    if cid:
        moa = get_moa_from_cid(cid)

    # If that fails and it looks like a CAS number, try the xref endpoint
    if not moa and is_cas_number(drug_str):
        cid = get_cid_by_cas(drug_str)
        if cid:
            moa = get_moa_from_cid(cid)

    cache[drug_str] = moa
    return moa

In [ ]:
moa_cache = {}
results = []

for drug in tqdm(missing_drugs, desc="Fetching MOAs"):
    moa = get_moa_for_drug(drug, moa_cache)
    results.append((drug, moa))
    time.sleep(0.2)


    

In [78]:
df_drugcomb['Drug1_upper'] = df_drugcomb['Drug1'].str.upper() 
df_drugcomb[df_drugcomb['Drug1_upper'] == '(+)-BICUCULLINE']

,ID,Drug1,Drug2,Cell line,ZIP,Bliss,Loewe,HSA,Durg1_Upper,Durg2_Upper,Drug1_upper
443978,443979,(+)-BICUCULLINE,TEMOZOLOMIDE,T98G,21.85,21.85,4.44,4.44,(+)-BICUCULLINE,TEMOZOLOMIDE,(+)-BICUCULLINE


In [74]:
results

[('KI-696', None),
 ('1265789-88-5', None),
 ('483-60-3', None),
 ('ADL5859 HCL', None),
 ('X-396', None),
 ('(+)-BICUCULLINE', None),
 ('RIVASTIGMINE TARTRATE', None)]

In [57]:
get_moa_pubchem('Chembl3348884')

{'source': 'PubChem', 'status': 'error', 'moa': "'PUGREST.BadRequest'"}

In [ ]:

results = []
print("Fetching MOA for each drug (PubChem first, ChEMBL fallback)...")

for drug in tqdm(unique_drugs):
    if drug in moa_cache:
        results.append({"drug": drug, **moa_cache[drug]})
        continue

    # Try PubChem first (fastest and most direct MOA)
    moa_info = get_moa_pubchem(drug)
    
    # If PubChem gives nothing useful, try ChEMBL
    if moa_info["status"] in ["not_found", "no_action"]:
        moa_info = get_moa_chembl(drug)
    
    entry = {"drug": drug, **moa_info}
    results.append(entry)
    moa_cache[drug] = moa_info
    
    # Save cache every 100 drugs
    if len(results) % 100 == 0:
        with open(CACHE_FILE, 'w', encoding='utf-8') as f:
            json.dump(moa_cache, f, ensure_ascii=False, indent=2)
    
    time.sleep(0.2)  # respect PubChem rate limit (~5 requests/sec)

# ====================== SAVE FINAL RESULTS ======================
final_df = pd.DataFrame(results)
final_df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Done! Saved {len(final_df):,} drugs with MOA to '{OUTPUT_CSV}'")
print(f"Cache saved to '{CACHE_FILE}' for future runs.")

# Optional: quick summary
print("\nMOA summary:")
print(final_df['moa'].value_counts().head(10))

In [2]:
!pip install chembl_webresource_client

  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
  Attempting uninstall: attrs
    Found existing installation: attrs 21.4.0
    Uninstalling attrs-21.4.0:
      Successfully uninstalled attrs-21.4.0


In [1]:
from chembl_webresource_client.new_client import new_client

In [16]:
!python -c "import attrs, cattrs; print(cattrs)"

<module 'cattrs' from '/villa/rhh25/Cellhit/venv/lib/python3.10/site-packages/cattrs/__init__.py'>


In [2]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
import time
import requests

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
# Ideally, load your drug list from the DrugComb dataset CSV.
# For this example, I will use a sample list of common cancer drugs.
# If you have the DrugComb dataset, load it like this:
# df_drugs = pd.read_csv('drugcomb_data.csv')
# drug_names = df_drugs['drug_name'].unique().tolist()

drug_names = [
    "Paclitaxel", "Doxorubicin", "Cisplatin", "5-Fluorouracil", 
    "Imatinib", "Erlotinib", "Venetoclax", "Trametinib"
]

OUTPUT_FILE = "drugcomb_moa_data.csv"

# Initialize ChEMBL client
molecule = new_client.molecule
target_client = new_client.target

def get_moa_from_chembl(drug_name):
    """
    Searches ChEMBL for a drug name and retrieves its Mechanism of Action.
    """
    try:
        # 1. Search for the molecule by name (filtering for approved drugs usually helps)
        # We only take the first match (most relevant)
        res = molecule.filter(molecule_synonyms__icontains=drug_name).only([
            'molecule_chembl_id', 
            'molecule_type', 
            'mechanism_of_action',
            'pref_name'
        ])
        
        found = False
        for mol in res:
            # Simple check to see if the synonym matches reasonably well
            # or if the preferred name matches
            if drug_name.lower() in str(mol.get('molecule_synonyms', [])).lower() or \
               drug_name.lower() == mol.get('pref_name', '').lower():
                
                chembl_id = mol['molecule_chembl_id']
                moa_list = mol.get('mechanism_of_action', [])
                
                # Extract MOA text
                if moa_list:
                    # Join multiple MOAs if they exist
                    moa_text = "; ".join([x.get('mechanism_of_action') for x in moa_list])
                else:
                    moa_text = "No specific MOA text found in ChEMBL."
                
                found = True
                return {
                    'Drug Name': drug_name,
                    'ChEMBL ID': chembl_id,
                    'MOA': moa_text
                }
        
        if not found:
            # Fallback: Try to fetch target info if MOA text is missing
            # This requires a separate call to get full details including targets
            return get_fallback_target_info(drug_name)

    except Exception as e:
        return {
            'Drug Name': drug_name,
            'ChEMBL ID': 'Error',
            'MOA': f"API Error: {str(e)}"
        }

def get_fallback_target_info(drug_name):
    """
    If explicit MOA text is missing, try to fetch the Target, 
    as Target usually implies Mechanism of Action.
    """
    try:
        # Search again to get the ID first
        res = molecule.filter(molecule_synonyms__icontains=drug_name)
        if not res:
            return {'Drug Name': drug_name, 'ChEMBL ID': 'Not Found', 'MOA': 'Drug not found in ChEMBL.'}
        
        chembl_id = res[0]['molecule_chembl_id']
        
        # Get the mechanism of action specifically (sometimes deeper in the object)
        # Or get the activity to infer target
        mechanism = molecule.get(chembl_id).get('mechanism_of_action')
        
        if mechanism:
            moa_str = "; ".join([m['mechanism_of_action'] for m in mechanism])
            return {'Drug Name': drug_name, 'ChEMBL ID': chembl_id, 'MOA': moa_str}
        else:
            return {
                'Drug Name': drug_name, 
                'ChEMBL ID': chembl_id, 
                'MOA': 'No explicit MOA. (Target lookup needed for full details)'
            }
            
    except Exception as e:
        return {
            'Drug Name': drug_name, 
            'ChEMBL ID': 'Error', 
            'MOA': f"Fallback Error: {str(e)}"
        }

# ---------------------------------------------------------
# EXECUTION
# ---------------------------------------------------------
results = []

print(f"Starting MOA retrieval for {len(drug_names)} drugs...")

for i, drug in enumerate(drug_names):
    print(f"[{i+1}/{len(drug_names)}] Processing: {drug}")
    
    data = get_moa_from_chembl(drug)
    results.append(data)
    
    # Be polite to the API (rate limiting)
    time.sleep(0.5)

# Save to CSV
df_output = pd.DataFrame(results)
df_output.to_csv(OUTPUT_FILE, index=False)

print(f"Done! Results saved to {OUTPUT_FILE}")
print(df_output.head())

Starting MOA retrieval for 8 drugs...
[1/8] Processing: Paclitaxel
[2/8] Processing: Doxorubicin
[3/8] Processing: Cisplatin
[4/8] Processing: 5-Fluorouracil
[5/8] Processing: Imatinib
[6/8] Processing: Erlotinib
[7/8] Processing: Venetoclax
[8/8] Processing: Trametinib
Done! Results saved to drugcomb_moa_data.csv
        Drug Name ChEMBL ID                                                MOA
0      Paclitaxel     Error  API Error: Error for url https://www.ebi.ac.uk...
1     Doxorubicin     Error  API Error: Error for url https://www.ebi.ac.uk...
2       Cisplatin     Error  API Error: Error for url https://www.ebi.ac.uk...
3  5-Fluorouracil     Error  API Error: Error for url https://www.ebi.ac.uk...
4        Imatinib     Error  API Error: Error for url https://www.ebi.ac.uk...


In [3]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
import time
import json

# ---------------------------------------------------------
# 1. LOAD DRUGCOMB DATA
# ---------------------------------------------------------
# Ideally, load your specific DrugComb file. 
# For this example, we will simulate a list of drugs found in DrugComb.
# If you have the file, use: df = pd.read_csv('drugcomb_data.csv')

# Example list of drugs commonly found in DrugComb
drugcomb_drugs = [
    "Erlotinib", "Gefitinib", "Afatinib", "Lapatinib", 
    "Paclitaxel", "Doxorubicin", "5-Fluorouracil", 
    "Methotrexate", "Cisplatin", "Venetoclax"
]

# ---------------------------------------------------------
# 2. SETUP CHEMBL CLIENT
# ---------------------------------------------------------
molecule = new_client.molecule

def get_moa_info(drug_name):
    """
    Queries ChEMBL for a specific drug name and extracts MOA data.
    """
    try:
        # Search for molecule by name (case-insensitive search)
        # We use molecule_name__icontains to find partial matches
        res = molecule.filter(molecule_name__icontains=drug_name)
        
        if not res:
            return None, None, "Not Found in ChEMBL"
        
        # Take the first match (usually the most relevant)
        chembl_entry = res[0]
        chembl_id = chembl_entry['molecule_chembl_id']
        print(f"Found {drug_name} in ChEMBL with ID: {chembl_id}")
        
        # Check for Mechanism of Action
        # The 'mechanism_of_action' field contains a list of dictionaries
        moa_list = chembl_entry.get('mechanism_of_action', [])
        
        if not moa_list:
            return chembl_id, [], "No MOA Data Listed"
            
        # Extract readable details from the MOA list
        formatted_moa = []
        for item in moa_list:
            # Typical structure: {'mechanism_of_action': 'inhibitor', 'target': 'CHEMBL240', ...}
            action = item.get('mechanism_of_action', 'Unknown Action')
            target = item.get('target', 'Unknown Target')
            formatted_moa.append(f"{action} (Target: {target})")
            
        return chembl_id, formatted_moa, "Success"

    except Exception as e:
        return None, None, f"Error: {str(e)}"

# ---------------------------------------------------------
# 3. ITERATE AND FETCH DATA
# ---------------------------------------------------------
results = []

print(f"Processing {len(drugcomb_drugs)} drugs from DrugComb...")

for i, drug in enumerate(drugcomb_drugs):
    print(f"[{i+1}/{len(drugcomb_drugs)}] Fetching data for: {drug}")
    
    chembl_id, moa_data, status = get_moa_info(drug)
    
    results.append({
        'DrugComb_Name': drug,
        'ChEMBL_ID': chembl_id,
        'Status': status,
        'MOA': "; ".join(moa_data) if moa_data else None
    })
    
    # Be polite to the API (avoid rate limiting)
    time.sleep(0.1)



Processing 10 drugs from DrugComb...
[1/10] Fetching data for: Erlotinib
Found Erlotinib in ChEMBL with ID: CHEMBL6329
[2/10] Fetching data for: Gefitinib
Found Gefitinib in ChEMBL with ID: CHEMBL6329
[3/10] Fetching data for: Afatinib
Found Afatinib in ChEMBL with ID: CHEMBL6329
[4/10] Fetching data for: Lapatinib
Found Lapatinib in ChEMBL with ID: CHEMBL6329
[5/10] Fetching data for: Paclitaxel
Found Paclitaxel in ChEMBL with ID: CHEMBL6329
[6/10] Fetching data for: Doxorubicin
Found Doxorubicin in ChEMBL with ID: CHEMBL6329
[7/10] Fetching data for: 5-Fluorouracil
Found 5-Fluorouracil in ChEMBL with ID: CHEMBL6329
[8/10] Fetching data for: Methotrexate
Found Methotrexate in ChEMBL with ID: CHEMBL6329
[9/10] Fetching data for: Cisplatin
Found Cisplatin in ChEMBL with ID: CHEMBL6329
[10/10] Fetching data for: Venetoclax
Found Venetoclax in ChEMBL with ID: CHEMBL6329


In [1]:
results

NameError: name 'results' is not defined

In [2]:
results

[{'DrugComb_Name': 'Erlotinib',
  'ChEMBL_ID': 'CHEMBL6329',
  'Status': 'No MOA Data Listed',
  'MOA': None},
 {'DrugComb_Name': 'Gefitinib',
  'ChEMBL_ID': 'CHEMBL6329',
  'Status': 'No MOA Data Listed',
  'MOA': None},
 {'DrugComb_Name': 'Afatinib',
  'ChEMBL_ID': 'CHEMBL6329',
  'Status': 'No MOA Data Listed',
  'MOA': None},
 {'DrugComb_Name': 'Lapatinib',
  'ChEMBL_ID': 'CHEMBL6329',
  'Status': 'No MOA Data Listed',
  'MOA': None},
 {'DrugComb_Name': 'Paclitaxel',
  'ChEMBL_ID': 'CHEMBL6329',
  'Status': 'No MOA Data Listed',
  'MOA': None},
 {'DrugComb_Name': 'Doxorubicin',
  'ChEMBL_ID': 'CHEMBL6329',
  'Status': 'No MOA Data Listed',
  'MOA': None},
 {'DrugComb_Name': '5-Fluorouracil',
  'ChEMBL_ID': 'CHEMBL6329',
  'Status': 'No MOA Data Listed',
  'MOA': None},
 {'DrugComb_Name': 'Methotrexate',
  'ChEMBL_ID': 'CHEMBL6329',
  'Status': 'No MOA Data Listed',
  'MOA': None},
 {'DrugComb_Name': 'Cisplatin',
  'ChEMBL_ID': 'CHEMBL6329',
  'Status': 'No MOA Data Listed',
  'MOA'

In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
import time
import requests

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
# Ideally, load your drug list from the DrugComb dataset CSV.
# For this example, I will use a sample list of common cancer drugs.
# If you have the DrugComb dataset, load it like this:
# df_drugs = pd.read_csv('drugcomb_data.csv')
# drug_names = df_drugs['drug_name'].unique().tolist()

drug_names = [
    "Paclitaxel", "Doxorubicin", "Cisplatin", "5-Fluorouracil", 
    "Imatinib", "Erlotinib", "Venetoclax", "Trametinib"
]

OUTPUT_FILE = "drugcomb_moa_data.csv"

# Initialize ChEMBL client
molecule = new_client.molecule
target_client = new_client.target

def get_moa_from_chembl(drug_name):
    """
    Searches ChEMBL for a drug name and retrieves its Mechanism of Action.
    """
    try:
        # 1. Search for the molecule by name (filtering for approved drugs usually helps)
        # We only take the first match (most relevant)
        res = molecule.filter(molecule_synonyms__icontains=drug_name).only([
            'molecule_chembl_id', 
            'molecule_type', 
            'mechanism_of_action',
            'pref_name'
        ])
        
        found = False
        for mol in res:
            # Simple check to see if the synonym matches reasonably well
            # or if the preferred name matches
            if drug_name.lower() in str(mol.get('molecule_synonyms', [])).lower() or \
               drug_name.lower() == mol.get('pref_name', '').lower():
                
                chembl_id = mol['molecule_chembl_id']
                moa_list = mol.get('mechanism_of_action', [])
                
                # Extract MOA text
                if moa_list:
                    # Join multiple MOAs if they exist
                    moa_text = "; ".join([x.get('mechanism_of_action') for x in moa_list])
                else:
                    moa_text = "No specific MOA text found in ChEMBL."
                
                found = True
                return {
                    'Drug Name': drug_name,
                    'ChEMBL ID': chembl_id,
                    'MOA': moa_text
                }
        
        if not found:
            # Fallback: Try to fetch target info if MOA text is missing
            # This requires a separate call to get full details including targets
            return get_fallback_target_info(drug_name)

    except Exception as e:
        return {
            'Drug Name': drug_name,
            'ChEMBL ID': 'Error',
            'MOA': f"API Error: {str(e)}"
        }

def get_fallback_target_info(drug_name):
    """
    If explicit MOA text is missing, try to fetch the Target, 
    as Target usually implies Mechanism of Action.
    """
    try:
        # Search again to get the ID first
        res = molecule.filter(molecule_synonyms__icontains=drug_name)
        if not res:
            return {'Drug Name': drug_name, 'ChEMBL ID': 'Not Found', 'MOA': 'Drug not found in ChEMBL.'}
        
        chembl_id = res[0]['molecule_chembl_id']
        
        # Get the mechanism of action specifically (sometimes deeper in the object)
        # Or get the activity to infer target
        mechanism = molecule.get(chembl_id).get('mechanism_of_action')
        
        if mechanism:
            moa_str = "; ".join([m['mechanism_of_action'] for m in mechanism])
            return {'Drug Name': drug_name, 'ChEMBL ID': chembl_id, 'MOA': moa_str}
        else:
            return {
                'Drug Name': drug_name, 
                'ChEMBL ID': chembl_id, 
                'MOA': 'No explicit MOA. (Target lookup needed for full details)'
            }
            
    except Exception as e:
        return {
            'Drug Name': drug_name, 
            'ChEMBL ID': 'Error', 
            'MOA': f"Fallback Error: {str(e)}"
        }

# ---------------------------------------------------------
# EXECUTION
# ---------------------------------------------------------
results = []

print(f"Starting MOA retrieval for {len(drug_names)} drugs...")

for i, drug in enumerate(drug_names):
    print(f"[{i+1}/{len(drug_names)}] Processing: {drug}")
    
    data = get_moa_from_chembl(drug)
    results.append(data)
    
    # Be polite to the API (rate limiting)
    time.sleep(0.5)

# Save to CSV
df_output = pd.DataFrame(results)
df_output.to_csv(OUTPUT_FILE, index=False)

print(f"Done! Results saved to {OUTPUT_FILE}")
print(df_output.head())

In [3]:
df = pd.read_csv("../../../dataset/drugcombs_scored.csv")

In [7]:
!pip install mygene

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 442.1 kB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
# Install: pip install mygene
import mygene

mg = mygene.MyGeneInfo()

# Example: You have a STRING Protein ID
protein_id = "ENSP00000234798"

# Query the database to find the corresponding Gene
result = mg.query(protein_id, fields='symbol', species='human')

# Extract the Gene Symbol
if result['hits']:
    gene_symbol = result['hits'][0]['symbol']
    print(f"Protein {protein_id} comes from Gene {gene_symbol}")
    # Output: Protein ENSP00000275493 comes from Gene EGFR

Input sequence provided is already in string format. No operation performed


Protein ENSP00000234798 comes from Gene TPSG1
